***Install dependencies***

In [1]:
!pip install requests 
!pip install beautifulsoup4 
!pip install lxml 
!pip install pandas 
!pip install tqdm

***Imports & helpers***

In [2]:
import os
import time
import random
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from bs4 import BeautifulSoup
import pandas as pd
from tqdm.notebook import tqdm


***Create a requests session with retries & polite User-Agent***

In [3]:
def create_session(user_agent=None, retries=3, backoff=0.4):
    s = requests.Session()
    ua = user_agent or (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    )
    s.headers.update({"User-Agent": ua})
    retry = Retry(
        total=retries,
        backoff_factor=backoff,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET", "POST"]
    )
    adapter = HTTPAdapter(max_retries=retry)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    return s

session = create_session()


***Check robots.txt***

In [4]:
def can_fetch(url, user_agent=None):
    """
    Return True if robots.txt allows fetching the URL for the session's UA.
    If robots.txt cannot be read, we warn and return True (still be polite).
    """
    ua = user_agent or session.headers.get("User-Agent", "*")
    parsed = urlparse(url)
    robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"
    rp = RobotFileParser()
    try:
        rp.set_url(robots_url)
        rp.read()
        return rp.can_fetch(ua, url)
    except Exception as e:
        print(f"Warning: couldn't read robots.txt for {parsed.netloc}: {e}")
        return True


***Robust fetcher returning BeautifulSoup + raw HTML***

In [5]:
def get_soup_and_html(url, timeout=12):
    """
    Use the session to GET the URL. Raises on HTTP errors.
    Returns (BeautifulSoup object, raw_html).
    """
    r = session.get(url, timeout=timeout)
    r.raise_for_status()
    html = r.text
    soup = BeautifulSoup(html, "lxml")
    return soup, html


***Parser for Hacker News listing page***

In [6]:
def parse_hn_listing_html(html, base_url="https://news.ycombinator.com/"):
    soup = BeautifulSoup(html, "lxml")
    rows = []
    for item in soup.select("tr.athing"):
        try:
            post_id = item.get("id")
            title_tag = item.select_one("a.storylink") or item.select_one("a.titlelink")
            title = title_tag.get_text(strip=True) if title_tag else None
            raw_link = title_tag.get("href") if title_tag else None
            link = urljoin(base_url, raw_link) if raw_link else None

            sub = item.find_next_sibling("tr").select_one("td.subtext")
            score = None; user = None; comments = None
            if sub:
                score_tag = sub.select_one("span.score")
                if score_tag:
                    try:
                        score = int(score_tag.get_text(strip=True).split()[0])
                    except:
                        score = None
                user_tag = sub.select_one("a.hnuser")
                user = user_tag.get_text(strip=True) if user_tag else None

                a_tags = sub.select("a")
                if a_tags:
                    last = a_tags[-1].get_text(strip=True)
                    if "comment" in last:
                        if last == "discuss":
                            comments = 0
                        else:
                            try:
                                comments = int(last.split()[0])
                            except:
                                comments = None

            rows.append({
                "post_id": post_id,
                "title": title,
                "link": link,
                "raw_link": raw_link,
                "score": score,
                "user": user,
                "comments": comments
            })
        except Exception as e:
            print("parse error:", e)
            continue
    return rows


***Main scraping loop — save raw pages & collect rows***

In [7]:
PAGES_TO_SCRAPE = 5   # one HN page ≈ 30 posts
OUTPUT_DIR = "scrape_output"
RAW_DIR = os.path.join(OUTPUT_DIR, "raw_pages")
os.makedirs(RAW_DIR, exist_ok=True)

all_rows = []
base_list_url = "https://news.ycombinator.com/news"

for page in tqdm(range(1, PAGES_TO_SCRAPE + 1), desc="HN pages"):
    url = f"{base_list_url}?p={page}"
    # Check robots.txt
    if not can_fetch(url):
        print(f"robots.txt forbids scraping {url} — skipping page {page}")
        continue

    try:
        soup, raw_html = get_soup_and_html(url)
    except Exception as e:
        print(f"Failed to GET {url}: {e}")
        continue

    # Save raw HTML
    raw_path = os.path.join(RAW_DIR, f"hn_page_{page}.html")
    with open(raw_path, "w", encoding="utf-8") as f:
        f.write(raw_html)

    # Parse and collect
    rows = parse_hn_listing_html(raw_html)
    all_rows.extend(rows)

    # Polite delay
    time.sleep(random.uniform(1.0, 2.4))

# Convert to DataFrame
df = pd.DataFrame(all_rows)
print("Total rows collected:", len(df))
df.head(8)


HN pages:   0%|          | 0/5 [00:00<?, ?it/s]

Total rows collected: 150


,post_id,title,link,raw_link,score,user,comments
0,45389267,None,None,None,338.0,kevlened,110.0
1,45388021,None,None,None,561.0,knowtheory,246.0
2,45391871,None,None,None,93.0,pykello,8.0
3,45392922,None,None,None,25.0,chmaynard,14.0
4,45391444,None,None,None,127.0,kristianp,16.0
5,45356429,None,None,None,151.0,andsoitis,25.0
6,45392164,None,None,None,69.0,exvi,12.0
7,45391566,None,None,None,87.0,TheFreim,112.0


***Save processed CSV + SQLite***

In [8]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
CSV_PATH = os.path.join(OUTPUT_DIR, "hn_metadata.csv")
df.to_csv(CSV_PATH, index=False)
print("Saved CSV:", CSV_PATH)

# Also save to SQLite
import sqlite3
DB_PATH = os.path.join(OUTPUT_DIR, "hn_data.db")
conn = sqlite3.connect(DB_PATH)
df.to_sql("hn_posts", conn, if_exists="replace", index=False)
conn.close()
print("Saved SQLite DB:", DB_PATH)


Saved CSV: scrape_output\hn_metadata.csv
Saved SQLite DB: scrape_output\hn_data.db


***fetch linked article text***

In [9]:
def simple_visible_text(soup, max_paragraphs=40):
    article = soup.find("article")
    if article:
        ps = article.find_all("p")
    else:
        main = soup.find("main")
        ps = main.find_all("p") if main else soup.find_all("p")
    texts = [p.get_text(separator=" ", strip=True) for p in ps[:max_paragraphs]]
    return " ".join([t for t in texts if t])

def fetch_article_text(url, max_chars=20000):
    if not url:
        return None
    parsed = urlparse(url)
    if parsed.netloc.endswith("ycombinator.com"):
        return None
    if not can_fetch(url):
        return None

    try:
        r = session.get(url, timeout=12)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "lxml")
        text = simple_visible_text(soup)
        if not text:
            p_tags = soup.find_all("p")
            text = " ".join([p.get_text(strip=True) for p in p_tags[:12]])
        return (text[:max_chars] if text else None)
    except Exception:
        return None

# Fetch article bodies for the first N unique links
N = 20
article_texts = []
sample_links = df['link'].dropna().unique()[:N]

for link in tqdm(sample_links, desc="fetching article bodies"):
    txt = fetch_article_text(link)
    article_texts.append({"link": link, "article_text": txt})
    time.sleep(random.uniform(1.0, 2.5))

art_df = pd.DataFrame(article_texts)
ART_CSV = os.path.join(OUTPUT_DIR, "hn_article_text_sample.csv")
art_df.to_csv(ART_CSV, index=False)
print("Saved article text sample to", ART_CSV)


fetching article bodies: 0it [00:00, ?it/s]

Saved article text sample to scrape_output\hn_article_text_sample.csv


***Quick EDA checks***

In [10]:
print("Total posts:", len(df))
print("Unique links:", df['link'].nunique())

# Top by score
display(df.sort_values("score", ascending=False).head(10))

# Score summary
print(df['score'].describe())

# Top users by number of posts
display(df['user'].value_counts().head(12))


Total posts: 150
Unique links: 0


,post_id,title,link,raw_link,score,user,comments
86,45374500,None,None,None,1051.0,Metalhearf,597.0
137,45362697,None,None,None,1035.0,bertman,541.0
106,45350690,None,None,None,841.0,alazsengul,466.0
135,45358940,None,None,None,695.0,_zie,119.0
139,45321849,None,None,None,676.0,foehrenwald,134.0
82,45373081,None,None,None,653.0,tosh,533.0
76,45375477,None,None,None,604.0,meetpateltech,696.0
138,45366867,None,None,None,594.0,spacebuffer,525.0
89,45347072,None,None,None,590.0,fibo,79.0
1,45388021,None,None,None,561.0,knowtheory,246.0


count     149.000000
mean      152.838926
std       196.310911
min         4.000000
25%        24.000000
50%        87.000000
75%       177.000000
max      1051.000000
Name: score, dtype: float64


user
todsacerdoti     8
meetpateltech    4
andsoitis        2
doener           2
redbell          2
zdw              2
ibobev           2
1vuio0pswjnm7    2
bookofjoe        2
Bender           2
transpute        2
klaussilveira    2
Name: count, dtype: int64

In [11]:
from pathlib import Path
import os, sys

print("Python executable:", sys.executable)
print("Working dir:", Path.cwd())
# Change this to the folder you expect the scraper to write to
expected_out = Path(r"C:\Users\HP\OneDrive\Desktop\CodeAlpha\scrape_output")
print("Expected OUTPUT_DIR exists?", expected_out.exists())
print("Contents of expected OUTPUT_DIR:", sorted([p.name for p in expected_out.glob("*")]) if expected_out.exists() else "No folder found")

# If your notebook used a relative OUTPUT_DIR variable, show files in notebook cwd
cwd_out = Path("scrape_output")
print("Relative scrape_output exists in cwd?", cwd_out.exists())
if cwd_out.exists():
    print("Files:", sorted([p.name for p in cwd_out.rglob("*")]))


Python executable: C:\Users\HP\AppData\Local\Programs\Python\Python313\python.exe
Working dir: C:\Users\HP\OneDrive\Desktop\CodeAlpha
Expected OUTPUT_DIR exists? True
Contents of expected OUTPUT_DIR: ['hn_article_text_sample.csv', 'hn_data.db', 'hn_metadata.csv', 'raw_pages']
Relative scrape_output exists in cwd? True
Files: ['hn_article_text_sample.csv', 'hn_data.db', 'hn_metadata.csv', 'hn_page_1.html', 'hn_page_2.html', 'hn_page_3.html', 'hn_page_4.html', 'hn_page_5.html', 'raw_pages']


In [12]:
from pathlib import Path
p = Path(r"C:\Users\HP\OneDrive\Desktop\CodeAlpha\scrape_output")
raw1 = p / "hn_page_1.html"
print("Exists?", raw1.exists())
print("Size (bytes):", raw1.stat().st_size if raw1.exists() else "N/A")
print("\nFirst 40 lines of the raw HTML (for inspection):\n")
if raw1.exists():
    with open(raw1, "r", encoding="utf-8") as f:
        for i in range(40):
            line = f.readline()
            if not line:
                break
            print(f"{i+1:02d}: {line.rstrip()}")
else:
    print("File not found:", raw1)


Exists? False
Size (bytes): N/A

First 40 lines of the raw HTML (for inspection):

File not found: C:\Users\HP\OneDrive\Desktop\CodeAlpha\scrape_output\hn_page_1.html


In [13]:
from pathlib import Path

# Point to raw_pages subfolder
RAW_DIR = Path(r"C:/Users/HP/OneDrive/Desktop/CodeAlpha/scrape_output/raw_pages")

# Check contents
print("Files inside RAW_DIR:", [f.name for f in RAW_DIR.glob("*.html")])


Files inside RAW_DIR: ['hn_page_1.html', 'hn_page_2.html', 'hn_page_3.html', 'hn_page_4.html', 'hn_page_5.html']


In [14]:
from pathlib import Path

# Correct folder where raw pages are saved
RAW_DIR = Path(r"C:/Users/HP/OneDrive/Desktop/CodeAlpha/scrape_output/raw_pages")

# Verify files exist
html_files = sorted(RAW_DIR.glob("hn_page_*.html"))
print("Found raw HTML pages:", len(html_files))
html_files[:5]  # show first few


Found raw HTML pages: 5


[WindowsPath('C:/Users/HP/OneDrive/Desktop/CodeAlpha/scrape_output/raw_pages/hn_page_1.html'),
 WindowsPath('C:/Users/HP/OneDrive/Desktop/CodeAlpha/scrape_output/raw_pages/hn_page_2.html'),
 WindowsPath('C:/Users/HP/OneDrive/Desktop/CodeAlpha/scrape_output/raw_pages/hn_page_3.html'),
 WindowsPath('C:/Users/HP/OneDrive/Desktop/CodeAlpha/scrape_output/raw_pages/hn_page_4.html'),
 WindowsPath('C:/Users/HP/OneDrive/Desktop/CodeAlpha/scrape_output/raw_pages/hn_page_5.html')]

In [16]:
from bs4 import BeautifulSoup

def parse_hn_listing_html_from_string(html: str):
    """
    Parse Hacker News listing HTML and extract article metadata.
    Returns a list of rows with:
    [rank, title, link, points, author, age, comments]
    """
    soup = BeautifulSoup(html, "html.parser")
    rows = []

    # each article is in a table row with class 'athing'
    for item in soup.select("tr.athing"):
        rank = item.select_one(".rank")
        title = item.select_one(".titleline a")
        subtext = item.find_next_sibling("tr").select_one(".subtext")

        # extract details
        rank = rank.text.strip().replace(".", "") if rank else None
        title_text = title.text.strip() if title else None
        link = title["href"] if title and title.has_attr("href") else None

        points = subtext.select_one(".score").text.split()[0] if subtext and subtext.select_one(".score") else None
        author = subtext.select_one(".hnuser").text if subtext and subtext.select_one(".hnuser") else None
        age = subtext.select_one(".age").text if subtext and subtext.select_one(".age") else None
        comments = subtext.find_all("a")[-1].text if subtext and subtext.find_all("a") else None

        rows.append([rank, title_text, link, points, author, age, comments])

    return rows


In [17]:
all_rows = []

for html_file in html_files:
    html = html_file.read_text(encoding="utf-8", errors="ignore")
    rows = parse_hn_listing_html_from_string(html)
    all_rows.extend(rows)

print("Total rows extracted:", len(all_rows))
all_rows[:5]


Total rows extracted: 150


[['1',
  'SimpleFold: Folding proteins is simpler than you think',
  'https://github.com/apple/ml-simplefold',
  '338',
  'kevlened',
  '12 hours ago',
  '110\xa0comments'],
 ['2',
  'Open Social',
  'https://overreacted.io/open-social/',
  '561',
  'knowtheory',
  '14 hours ago',
  '246\xa0comments'],
 ['3',
  "New math revives geometry's oldest problems",
  'https://www.quantamagazine.org/new-math-revives-geometrys-oldest-problems-20250926/',
  '93',
  'pykello',
  '7 hours ago',
  '8\xa0comments'],
 ['4',
  "Why Today's Humanoids Won't Learn Dexterity",
  'https://rodneybrooks.com/why-todays-humanoids-wont-learn-dexterity/',
  '25',
  'chmaynard',
  '3 hours ago',
  '14\xa0comments'],
 ['5',
  'Moondream 3 Preview: Frontier-level reasoning at a blazing speed',
  'https://moondream.ai/blog/moondream-3-preview',
  '127',
  'kristianp',
  '8 hours ago',
  '16\xa0comments']]

In [18]:
df = pd.DataFrame(all_rows, columns=["Rank", "Title", "Link", "Points", "Author", "Age", "Comments"])
print("DataFrame shape:", df.shape)
df.head(10)


DataFrame shape: (150, 7)


,Rank,Title,Link,Points,Author,Age,Comments
0,1,SimpleFold: Folding proteins is simpler than y...,https://github.com/apple/ml-simplefold,338,kevlened,12 hours ago,110 comments
1,2,Open Social,https://overreacted.io/open-social/,561,knowtheory,14 hours ago,246 comments
2,3,New math revives geometry's oldest problems,https://www.quantamagazine.org/new-math-revive...,93,pykello,7 hours ago,8 comments
3,4,Why Today's Humanoids Won't Learn Dexterity,https://rodneybrooks.com/why-todays-humanoids-...,25,chmaynard,3 hours ago,14 comments
4,5,Moondream 3 Preview: Frontier-level reasoning ...,https://moondream.ai/blog/moondream-3-preview,127,kristianp,8 hours ago,16 comments
5,6,The Beauty of Programming (2001),https://www.brynmawr.edu/inside/academic-infor...,151,andsoitis,9 hours ago,25 comments
6,7,The Obsessively Complete Infocom Catalog,https://eblong.com/infocom/,69,exvi,6 hours ago,12 comments
7,8,Thoughts on Mechanical Keyboards and the ZSA M...,https://www.masteringemacs.org/article/thought...,87,TheFreim,7 hours ago,112 comments
8,9,If you are harassed by lasers,https://www.laserpointersafety.com/harassment....,147,1970-01-01,11 hours ago,140 comments
9,10,CT scans of 1k lithium-ion batteries show qual...,https://www.lumafield.com/article/finding-hidd...,165,jonbruner,14 hours ago,54 comments


In [19]:
OUTPUT_CSV = Path(r"C:\Users\HP\OneDrive\Desktop\CodeAlpha\scrape_output\hn_metadata_rebuilt.csv")

df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print("Saved dataset to:", OUTPUT_CSV)


Saved dataset to: C:\Users\HP\OneDrive\Desktop\CodeAlpha\scrape_output\hn_metadata_rebuilt.csv


In [20]:
check_df = pd.read_csv(OUTPUT_CSV)
print("Reloaded shape:", check_df.shape)
check_df.head(5)


Reloaded shape: (150, 7)


,Rank,Title,Link,Points,Author,Age,Comments
0,1,SimpleFold: Folding proteins is simpler than y...,https://github.com/apple/ml-simplefold,338.0,kevlened,12 hours ago,110 comments
1,2,Open Social,https://overreacted.io/open-social/,561.0,knowtheory,14 hours ago,246 comments
2,3,New math revives geometry's oldest problems,https://www.quantamagazine.org/new-math-revive...,93.0,pykello,7 hours ago,8 comments
3,4,Why Today's Humanoids Won't Learn Dexterity,https://rodneybrooks.com/why-todays-humanoids-...,25.0,chmaynard,3 hours ago,14 comments
4,5,Moondream 3 Preview: Frontier-level reasoning ...,https://moondream.ai/blog/moondream-3-preview,127.0,kristianp,8 hours ago,16 comments
